In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import copy
from scipy.stats import wilcoxon, mannwhitneyu
import pickle as pkl

import nibabel as nib

import matplotlib.pyplot as plt 

from sklearn.model_selection import train_test_split

from sklearn.feature_selection import SelectFromModel
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_regression, mutual_info_regression

from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor

from sklearn.metrics import mean_squared_error, r2_score, median_absolute_error, mean_absolute_error
from sklearn import linear_model

from pytorch_tabnet.tab_model import TabNetRegressor

import xgboost as xgb

In [ ]:
def load_unet_result(path, _print):
    unet_df = pd.read_csv(path, index_col = 'Unnamed: 0')
    index_values = []
    for _index in unet_df.index:
        if _index == " ":
            continue
        index_values.append(_index.split('-seg')[0]  # strip '-seg' suffix added by U-Net inference script)

    unet_df.index = index_values  
    unet_df.drop(['WT jaccard', 'TC jaccard', 'ET jaccard']  # Dice is the primary metric; Jaccard unused, axis = 1, inplace = True)
    summary_unet_df = pd.DataFrame(zip(unet_df.mean().values.tolist(), 
                                       unet_df.std().values.tolist()), 
                                   columns = ['mean', 'std'], index = unet_df.columns)
    if _print:
        print("****UNet******")
        print(summary_unet_df)
    return summary_unet_df, unet_df

def load_nnunet_result(path, _print):
    with open(path, 'r') as file:
        data = json.load(file)

    WT = []
    TC = []
    ET = []
    file_name = []
    for case in data['metric_per_case']:
        WT.append(case['metrics']['(2, 1, 3)']['Dice'])
        TC.append(case['metrics']['(2, 3)']['Dice'])
        ET.append(case['metrics']['(3,)']['Dice'])
        file_name.append(case['reference_file'].split('/')[-1].split('.')[0])

    nnunet_df = pd.DataFrame(zip(WT, TC, ET), columns = ['WT dice', 'TC dice', 'ET dice'], 
                             index = file_name)
    summary_nnunet_df = pd.DataFrame(zip(nnunet_df.mean().values.tolist(), 
                                       nnunet_df.std().values.tolist()), 
                                   columns = ['mean', 'std'], index = nnunet_df.columns)
    if _print:
        print("****nnUNet******")
        print(summary_nnunet_df)
    return summary_nnunet_df, nnunet_df

def load_TransBTS_result(path, _print):
    with open(path, 'r') as file:
        data = json.load(file)

    WT = []
    TC = []
    ET = []
    file_name = []
    for case_id in data.keys():
        case = data[case_id]
        WT.append(case['WT'][0])
        TC.append(case['TC'][0])
        ET.append(case['ET'][0])
        file_name.append(case_id)

    TransBTS_df = pd.DataFrame(zip(WT, TC, ET), columns = ['WT dice', 'TC dice', 'ET dice'], 
                             index = file_name)
    summary_TransBTS_df = pd.DataFrame(zip(TransBTS_df.mean().values.tolist(), 
                                       TransBTS_df.std().values.tolist()), 
                                   columns = ['mean', 'std'], index = TransBTS_df.columns)
    if _print:
        print("****TransBTS******")
        print(summary_TransBTS_df)
    return summary_TransBTS_df, TransBTS_df

In [ ]:
def read_results(_print=True):
    path = '../Results/Result/Vanilla_Unet/Unet_test_dice.csv'
    summary_unet_df, unet_df = load_unet_result(path, _print)

    path = '../Results/Result/nnUnet/nnUNetTrainer/summary.json'
    summary_da_nnunet_df, nnunet_da_df = load_nnunet_result(path, _print)

    path = '../Results/Result/nnUnet/nnUNetTrainerNoDA/summary.json'
    summary_noda_nnunet_df, nnunet_noda_df = load_nnunet_result(path, _print)

    path = '../Results/Result/TransBTS/submission/TransBTS2023-11-03/TransBTS_summary.json'
    summary_TransBTS_df, TransBTS_df = load_TransBTS_result(path, _print)
    return unet_df, nnunet_noda_df, nnunet_da_df, TransBTS_df




In [ ]:
def get_overlaps(unet_df, TransBTS_df, nnunet_noda_df, WT_dice_threshold, TC_dice_threshold, ET_dice_threshold):
    unet_df_sub = unet_df[(unet_df['WT dice'] < WT_dice_threshold) 
                          & (unet_df['TC dice'] < TC_dice_threshold) 
                          & (unet_df['ET dice'] < ET_dice_threshold)]
    
    TransBTS_df_sub = TransBTS_df[(TransBTS_df['WT dice'] < WT_dice_threshold) 
                          & (TransBTS_df['TC dice'] < TC_dice_threshold) 
                          & (TransBTS_df['ET dice'] < ET_dice_threshold)]
    
    nnunet_noda_df_sub = nnunet_noda_df[(nnunet_noda_df['WT dice'] < WT_dice_threshold) 
                          & (nnunet_noda_df['TC dice'] < TC_dice_threshold) 
                          & (nnunet_noda_df['ET dice'] < ET_dice_threshold)]

    unet_df_sub_subjects = unet_df_sub.index.values.tolist()
    TransBTS_df_sub_subjects = TransBTS_df_sub.index.values.tolist()
    nnunet_noda_df_sub_subjects = nnunet_noda_df_sub.index.values.tolist()

    all_overlaps = list(set(unet_df_sub_subjects) & set(TransBTS_df_sub_subjects) & set(nnunet_noda_df_sub_subjects))
#     print('all overlap', len(all_overlaps), unet_df_sub.shape, nnunet_noda_df_sub.shape)

    unet_nnunet_overlaps = list(set(unet_df_sub_subjects) & set(nnunet_noda_df_sub_subjects))
#     print('unet-nnunet overlap', len(unet_nnunet_overlaps))

    unet_TransBTS_overlaps = list(set(unet_df_sub_subjects) & set(TransBTS_df_sub_subjects))
#     print('unet-TransBTS overlap', len(unet_TransBTS_overlaps))

    nnunet_TransBTS_overlaps = list(set(TransBTS_df_sub_subjects) & set(nnunet_noda_df_sub_subjects))
#     print('nnunet-TransBTS overlap', len(nnunet_TransBTS_overlaps))
    
    return all_overlaps, unet_nnunet_overlaps

In [ ]:
def read_radiomics_results(analysis_type, location):
    file_name = '../Results/Analysis_Results/Radiomics/' + location + '/' + analysis_type + '.pkl'
    with open(file_name, 'rb') as f:
        results = pkl.load(f)
    return results

def read_MRI(dataset, patient_id):
    if dataset == 'Brats2020':
        baseloc = '../input/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/'
        pefix = 'BraTS20_Training_' + patient_id + '/' + 'BraTS20_Training_' + patient_id
        suffixs = ['_flair.nii','_t2.nii', '_t1.nii', '_t1ce.nii', '_seg.nii']
    elif dataset == 'Brats2023':
        baseloc = '../input/Brats2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/'
        pefix = patient_id + '/' + patient_id
        suffixs = ['-t2f.nii.gz','-t2w.nii.gz', '-t1n.nii.gz', '-t1c.nii.gz', '-seg.nii.gz']

    flair_filename = baseloc + pefix + suffixs[0]
    flair_img_f = nib.load(flair_filename)
    flair_img = np.asarray(flair_img_f.dataobj)

    t2_filename = baseloc + pefix + suffixs[1]
    t2_img_f = nib.load(t2_filename)
    t2_img = np.asarray(t2_img_f.dataobj)

    t1_filename = baseloc + pefix + suffixs[2]
    t1_img_f = nib.load(t1_filename)
    t1_img = np.asarray(t1_img_f.dataobj)

    t1ce_filename = baseloc + pefix + suffixs[3]
    t1ce_img_f = nib.load(t1ce_filename)
    t1ce_img = np.asarray(t1ce_img_f.dataobj)
    
    mask_filename = baseloc + pefix + suffixs[4]
    mask_img_f = nib.load(mask_filename)
    mask_img = np.asarray(mask_img_f.dataobj)
    
    return flair_img, t2_img, t1_img, t1ce_img, mask_img 

def preprocess_mask_labels(mask):
    # whole tumour
    mask_WT = mask.copy()
    mask_WT[mask_WT == 1] = 1
    mask_WT[mask_WT == 2] = 1
    mask_WT[mask_WT == 3] = 1
    # include all tumours 

    # NCR / NET - LABEL 1
    mask_TC = mask.copy()
    mask_TC[mask_TC == 1] = 1
    mask_TC[mask_TC == 2] = 0
    mask_TC[mask_TC == 3] = 1
    # exclude 2 / 4 labelled tumour 

    # ET - LABEL 4 
    mask_ET = mask.copy()
    mask_ET[mask_ET == 1] = 0
    mask_ET[mask_ET == 2] = 0
    mask_ET[mask_ET == 3] = 1
    # exclude 2 / 1 labelled tumour 

    mask = np.stack([mask_WT, mask_TC, mask_ET])
    
    return mask 

def read_mask_file(patient_id):
    baseloc = '../input/Brats2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/'
    pefix = patient_id + '/' + patient_id
    suffixs = ['-t2f.nii.gz','-t2w.nii.gz', '-t1n.nii.gz', '-t1c.nii.gz', '-seg.nii.gz']
    outputloc = '../Results/Result/Vanilla_Unet/' + patient_id
    
    sample_filename_mask = baseloc + pefix + suffixs[4]
    sample_mask_f = nib.load(sample_filename_mask)
    sample_mask = np.asarray(sample_mask_f.dataobj)
    
    masks = preprocess_mask_labels(sample_mask)
    mask_WT, mask_TC, mask_ET = masks[0], masks[1], masks[2]
    
    return mask_WT

def remove_duplicate_columns(df):
    """
    Remove columns from a DataFrame that have identical values to other columns, regardless of the column name.
    
    Parameters:
    - df: The pandas DataFrame from which to remove duplicate columns.
    
    Returns:
    - A new DataFrame with duplicate columns removed.
    """
    columns_to_remove = set()
    for i in range(df.shape[1]): # Iterate over all columns
        for j in range(i + 1, df.shape[1]): # Compare each column with every other column
            # Check if not already identified as duplicate and if equal
            if i not in columns_to_remove and j not in columns_to_remove:
                if df.iloc[:, i].equals(df.iloc[:, j]):
                    columns_to_remove.add(j)
    
    # Create a new DataFrame without the duplicate columns
    df_cleaned = df.drop(columns=df.columns[list(columns_to_remove)])
    return df_cleaned

def get_dataset(performance_df, analysis_types, location):
    results_df = performance_df
    for analysis_type in analysis_types:
        try:
            radiomics_results_df = read_radiomics_results(analysis_type, location)
        except Exception as e:
            print(e)
            continue
        properties = {}
        property_df = pd.DataFrame()
        for i in range(len(radiomics_results_df.keys())):
            key = list(radiomics_results_df.keys())[i]
            properties[i] = key

        for i in range(len(properties)):
            selected_property = properties[i]

            property_result_df = pd.DataFrame.from_dict(radiomics_results_df[selected_property], 
                                                        orient = 'index').astype(float)
            new_col = []
            for col in property_result_df.columns:
                new_col.append(selected_property + '_' + col + '_' + analysis_type)
            property_result_df.columns = new_col
            results_df = pd.merge(property_result_df, 
                           results_df, 
                           left_index=True, 
                           right_index=True)
#             print(property_result_df.shape)

#     print(results_df)
    remove_index = results_df.sort_values(['WT dice'])[0:20].index
    low_examples = results_df.loc[remove_index]
    results_df.drop(['WT dice', 'TC dice', 'ET dice'], axis=1, inplace=True)
    # results_df.drop(remove_index, axis = 0, inplace=True)
    
    return results_df, low_examples

def remove_duplicate_columns(df):
    """
    Remove columns from a DataFrame that have identical values to other columns, regardless of the column name.
    
    Parameters:
    - df: The pandas DataFrame from which to remove duplicate columns.
    
    Returns:
    - A new DataFrame with duplicate columns removed.
    """
    columns_to_remove = set()
    for i in range(df.shape[1]): # Iterate over all columns
        for j in range(i + 1, df.shape[1]): # Compare each column with every other column
            # Check if not already identified as duplicate and if equal
            if i not in columns_to_remove and j not in columns_to_remove:
                if df.iloc[:, i].equals(df.iloc[:, j]):
                    columns_to_remove.add(j)
    
    # Create a new DataFrame without the duplicate columns
    df_cleaned = df.drop(columns=df.columns[list(columns_to_remove)])
    return df_cleaned



def select_features(results_df, location):
    summary_df = pd.read_csv('../Results/Analysis_Results/Radiomics/summary/' + location + '_summary.csv')
    
    summary_df['property'] = summary_df['property'] + '_' +summary_df['Unnamed: 0'] + '_' +summary_df['analysis_type']

    important_features = summary_df[(summary_df['Statistically_Different'] != '-') 
                                    & ((summary_df['Effect_Size'] == 'Large')
                                       | (summary_df['Effect_Size'] == 'Medium'))].property.values.tolist()

    for feature in results_df.columns:
        if feature not in important_features:
            results_df.drop([feature], inplace=True, axis=1)
#     results_df = results_df[important_features]
    results_df = remove_duplicate_columns(results_df)
    # low_examples = low_examples[important_features]
    return results_df

In [ ]:
# Thresholds from paper: cases below all three are concordant-poor
WT_dice_threshold= 0.91
TC_dice_threshold= 0.86
ET_dice_threshold= 0.85

unet_df, nnunet_noda_df, nnunet_da_df, TransBTS_df = read_results(False)
performance_df = unet_df

all_overlaps, unet_nnunet_overlaps = get_overlaps(unet_df, 
                                                TransBTS_df, 
                                                nnunet_noda_df, 
                                                WT_dice_threshold, 
                                                TC_dice_threshold, 
                                                ET_dice_threshold)

# Oracle Model — Gradient Boosting Regression

Trains a Gradient Boosting Regressor to predict per-case WT Dice scores from handcrafted radiomic features using 5-fold cross-validation.

In [ ]:
analysis_types = ['firstorder', 'shape' , 'size',
                  'glcm_1', 'glcm_5', 'glcm_10', 
                  'gldm_1', 'gldm_5', 'gldm_10', 
                  'glrlm', 'glszm', 'intensity', 
                  'ngtdm_1', 'ngtdm_5','ngtdm_10']

# analysis_types = ['firstorder', 'shape' , 'size',
#                 'intensity']

location = 'Tumor_WT'

Tumor_results_df, low_examples = get_dataset(performance_df, analysis_types, location)
# Tumor_results_df = select_features(Tumor_results_df, location)

Tumor_results_df.shape

# Radiomics Non Tumor

In [ ]:
analysis_types = ['firstorder', 'shape' , 'size',
                  'glcm_1', 'gldm_1', 
                  'glrlm', 'glszm', 'intensity', 
                  'ngtdm_1']

# analysis_types = ['firstorder', 'shape' , 'size',
#                 'intensity']

location = 'Non_Tumor'

Non_Tumor_results_df, low_examples = get_dataset(performance_df, analysis_types, location)
Non_Tumor_results_df = select_features(Non_Tumor_results_df, location)

Non_Tumor_results_df.shape

# Radiomics Boundary

In [ ]:
analysis_types = ['firstorder', 'shape' , 'size',
                  'glcm_1', 'glcm_5', 
                  'gldm_1', 'gldm_5', 
                  'glrlm', 'glszm', 'intensity', 
                  'ngtdm_1', 'ngtdm_5']

# analysis_types = ['firstorder', 'shape' , 'size',
#                 'intensity']

location = 'Boundary'

Boundary_results_df, low_examples = get_dataset(performance_df, analysis_types, location)
Boundary_results_df = select_features(Boundary_results_df, location)

Boundary_results_df.shape

# Other Extracted Features

In [ ]:
volume_df = pd.read_csv('../Results/Analysis_Results/volume/GLI-Tumor_volumns.csv', 
                         index_col='Unnamed: 0')



Boundary_results_df = pd.read_csv('../Results/Analysis_Results/intensity/GLI-Image_intensity.csv', index_col='Unnamed: 0')

In [ ]:
results_df = pd.merge(Tumor_results_df, 
                       Non_Tumor_results_df, 
                       left_index=True, 
                       right_index=True)

results_df = pd.merge(results_df, 
                       Boundary_results_df, 
                       left_index=True, 
                       right_index=True)

results_df = pd.merge(results_df, 
                       performance_df, 
                       left_index=True, 
                       right_index=True)

results_df = remove_duplicate_columns(results_df)
# results_df.head()

In [ ]:
results_df.shape

In [ ]:
# Separate features and target variable
X = results_df.drop(['WT dice', 'TC dice', 'ET dice'], axis=1)  # Features
y = results_df['WT dice']  # Target variable (Dice score)

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=56)

# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# # Initialize the Gradient Boosting Regressor
model = GradientBoostingRegressor(n_estimators=100, 
                                  learning_rate=0.05, 
                                  max_depth=3, 
                                  random_state=42)

# model = SVR(C=6.0, epsilon=0.05)


# model = linear_model.BayesianRidge()

# # Initialize the RandomForestRegressor
# model = RandomForestRegressor(n_estimators = 100, 
#                               max_depth=5, 
#                               random_state=42, 
#                               criterion='absolute_error')

# # Initialize and fit the XGBRegressor
# model = xgb.XGBRegressor(objective='reg:squarederror', 
#                          n_estimators=200, 
#                          learning_rate=0.1, 
#                          max_depth=15, 
#                          subsample = 0.5,
#                          random_state=42)


# Train the model
model.fit(X_train_scaled, y_train)

# Predict on the testing set
y_pred = model.predict(X_test_scaled)

# Evaluate the model
mse = round(mean_squared_error(y_test, y_pred),2)
medae = round(median_absolute_error(y_test, y_pred),3)
mae = round(mean_absolute_error(y_test, y_pred),2)
r2 = round(r2_score(y_test, y_pred),2)


print(f"Mean Squared Error (MSE): {mse}")
print(f"Median Absolute Error (MedAE): {medae}")
print(f"Mean Absolute Error (MAE): {mae}")
print(f"R-squared (R2): {r2}")


In [ ]:
X_test['WT dice'] = y_test.values
X_test['predicted WT dice'] = y_pred
X_test['difference'] = abs(y_test.values - y_pred)
X_test = X_test.round(2)

In [ ]:
X_test

In [ ]:
wrong_cases = X_test[(X_test['WT dice'] >= WT_dice_threshold) 
                     & (X_test['predicted WT dice'] < WT_dice_threshold)]

all_cases = X_test[(X_test['WT dice'] >= WT_dice_threshold)]

borderline_cases = wrong_cases[wrong_cases['difference'] <= mae]

reduce = wrong_cases[wrong_cases['difference'] <= mae].shape[0]

round(1 - ((wrong_cases.shape[0]-reduce)/all_cases.shape[0]),2), all_cases.shape[0], wrong_cases.shape[0]

In [ ]:
wrong_cases[~wrong_cases.index.isin(borderline_cases.index)]

In [ ]:
wrong_cases = X_test[(X_test['WT dice'] < WT_dice_threshold) 
                     & (X_test['predicted WT dice'] >= WT_dice_threshold)]

all_cases = X_test[(X_test['WT dice'] <  WT_dice_threshold)]

borderline_cases = wrong_cases[wrong_cases['difference'] <= mae]
reduce = wrong_cases[wrong_cases['difference'] <= mae].shape[0]

round(1 - ((wrong_cases.shape[0]-reduce)/all_cases.shape[0]),2), all_cases.shape[0], wrong_cases.shape[0]

In [ ]:
wrong_cases[~wrong_cases.index.isin(borderline_cases.index)]

In [ ]:
failed_cases = X_test[(X_test['WT dice'] < WT_dice_threshold) 
                      & (X_test['predicted WT dice'] >= WT_dice_threshold) 
                      & (wrong_cases['difference'] > mae)]

print(round(1 - failed_cases.shape[0]/X_test.shape[0], 2))

In [ ]:
# X_test[X_test['predicted WT dice'] < WT_dice_threshold].shape[0]

In [ ]:
# X_test[(X_test['WT dice'] < WT_dice_threshold) 
#                      & (X_test['predicted WT dice'] < WT_dice_threshold)].shape[0]

In [ ]:
# X_test['WT dice'] = y_test.values
# X_test['predicted WT dice'] = y_pred
# X_test['difference'] = abs(y_test.values - y_pred)
# X_test['predicted WT dice reduced'] = X_test['predicted WT dice'] - mae

# X_test = X_test.round(2)

# X_test[(X_test['predicted WT dice reduced'] < WT_dice_threshold) 
#        & (X_test['predicted WT dice'] >= WT_dice_threshold)]



# TabNet

In [ ]:
import torch
from pytorch_tabnet.tab_model import TabNetClassifier

# Separate features and target variable
X = results_df.drop(['WT dice', 'TC dice', 'ET dice'], axis=1)  # Features
y = results_df['WT dice']  # Target variable (Dice score)

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)




# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_val_scaled = scaler.transform(X_val)

X_train_scaled_tensor = torch.from_numpy(X_train_scaled).float()
X_test_scaled_tensor = torch.from_numpy(X_test_scaled).float()
X_val_scaled_tensor = torch.from_numpy(X_val_scaled).float()

y_train_tensor = torch.from_numpy(y_train.values.reshape(-1, 1)).float()
y_test_tensor = torch.from_numpy(y_test.values.reshape(-1, 1)).float()
y_val_tensor = torch.from_numpy(y_val.values.reshape(-1, 1)).float()

tabnet_params = dict(
    n_d=32, 
    n_a=32, 
    n_steps=6,
    gamma=1.2,
    n_independent=2,
    n_shared=2,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2),
    mask_type='entmax', # "sparsemax"
    scheduler_params=dict(mode="min",
                          patience=5,
                          min_lr=1e-5,
                          factor=0.9,),
    scheduler_fn=torch.optim.lr_scheduler.ReduceLROnPlateau,
    verbose=10
)

model = TabNetRegressor(**tabnet_params)

model.fit(
    X_train=X_train_scaled_tensor.numpy(), y_train=y_train_tensor.numpy(),
    eval_set=[(X_val_scaled_tensor.numpy(), y_val_tensor.numpy())],
    eval_name=['test'],
    eval_metric=['mae'],
    max_epochs=1000,
    patience=50, # Early stopping patience
    batch_size=1024, virtual_batch_size=128,
    num_workers=0,
    drop_last=False
)

# Predict on the testing set
y_pred = model.predict(X_test_scaled_tensor.numpy())

# Convert predictions to tensor for evaluation
y_pred_tensor = torch.from_numpy(y_pred).float()

# Since TabNet's predict method returns numpy array, convert y_test to numpy for evaluation
y_test_numpy = y_test.values

# Evaluate the model
mse = mean_squared_error(y_test_numpy, y_pred_tensor.numpy())
mae = mean_absolute_error(y_test_numpy, y_pred_tensor.numpy())
r2 = r2_score(y_test_numpy, y_pred_tensor.numpy())


print(f"Mean Squared Error (MSE): {mse}")
print(f"Mean Absolute Error (MAE): {mae}")
print(f"R-squared (R2): {r2}")


In [ ]:
X_test['WT dice'] = y_test_numpy
X_test['predicted WT dice'] = y_pred_tensor.numpy()
X_test = X_test.round(2)

In [ ]:
wrong_cases = X_test[(X_test['WT dice'] < WT_dice_threshold) 
                     & (X_test['predicted WT dice'] < WT_dice_threshold)]

all_cases = X_test[(X_test['WT dice'] <  WT_dice_threshold)]

# reduce = wrong_cases[wrong_cases['difference'] <= mae].shape[0]

round(1 - ((wrong_cases.shape[0]-reduce)/all_cases.shape[0]),2), all_cases.shape[0], wrong_cases.shape[0]

In [ ]:
X_test.loc['BraTS-GLI-01610-000','predicted WT dice']